# 03 — Baseline Models (ARIMA / SARIMA / Naive)
Establish baseline performance. Any ML/DL model must beat these — if it doesn't, something is wrong.

In [ ]:
import sys
sys.path.insert(0, '..')

import pandas as pd
import warnings
warnings.filterwarnings('ignore')

from src.features import build_feature_matrix
from src.backtest import walk_forward_backtest
from src.evaluate import compile_metrics, save_metrics, plot_predictions
from src.models.baseline import ARIMAWrapper, SARIMAWrapper, NaiveLastValue

df_raw = pd.read_csv('../data/raw/macro_raw.csv', index_col='date', parse_dates=True)
print('Loaded raw data')

In [ ]:
# Run baselines for CPI (monthly, most data → fastest to test)
target = 'cpi'
results = []

for horizon in [1, 2, 4]:
    X, y = build_feature_matrix(df_raw, target_col=target, forecast_horizon=horizon)
    
    for ModelClass, name in [
        (NaiveLastValue, 'Naive'),
        (lambda: ARIMAWrapper(order=(2,0,1)), 'ARIMA(2,0,1)'),
        (lambda: SARIMAWrapper(order=(1,0,1), seasonal_order=(1,0,1,12)), 'SARIMA'),
    ]:
        model = ModelClass() if callable(ModelClass) else ModelClass
        print(f'  Running {name} h={horizon}...')
        result = walk_forward_backtest(X, y, model, model_name=name, horizon=horizon, min_train_size=60)
        results.append(result)
        print(f'    RMSE={result.rmse:.4f}  MAE={result.mae:.4f}')

metrics_df = compile_metrics(results)
print('\n── Results ──────────────────────────')
print(metrics_df.to_string(index=False))

In [ ]:
# Plot predictions for best baseline
best = min(results, key=lambda r: r.rmse)
print(f'Best baseline: {best.model_name} h={best.horizon} RMSE={best.rmse:.4f}')
plot_predictions(best, title=f'Best Baseline — {target.upper()}')

In [ ]:
save_metrics(metrics_df, path=None)  # saves to outputs/metrics.csv
print('Baseline metrics saved. Move to 04_ml_dl_models.ipynb')